<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_01_gru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_01 - TUNING - GRU**

```python
model = 'gru_balanced'
windows_size = [30, 60, 180]
targets = [t2_dir_thr_90, t2_dir_thr_120]
```

Hiperparámetros clave:

* hidden_size
* num_layers
* dropout
* learning_rate
* batch_size

Espacio sugerido:

```python
gru_params = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.2, 0.3],
    "learning_rate": [1e-3, 5e-4, 1e-4],
    "batch_size": [1024, 2048]
}
```



Vamos a hacer un **tuneo grueso** de la GRU, no conviene abrir demasiados hiperparámetros al mismo tiempo. En esta etapa inicial, lo correcto es priorizar los que más mueven el equilibrio entre **capacidad del modelo, regularización y estabilidad de entrenamiento**, porque en datos financieros el riesgo de sobreajuste es alto y el objetivo es encontrar configuraciones que generalicen bien fuera de muestra.   

a) Hiperparámetros que conviene tunear primero

Empezaremos con estos cinco:

* `hidden_size`
* `num_layers`
* `dropout`
* `learning_rate`
* `weight_decay`

b) Por qué estos primero

* `hidden_size`: controla la capacidad de representación de la GRU. Si es muy chico, subajusta; si es muy grande, sobreajusta.
* `num_layers`: controla profundidad secuencial. Suele influir bastante, pero no conviene abrir demasiados valores al inicio.
* `dropout`: es una de las regularizaciones más directas para redes profundas.
* `learning_rate`: suele ser uno de los hiperparámetros más sensibles en entrenamiento por gradiente.
* `weight_decay`: añade regularización sobre los pesos y ayuda a controlar complejidad, algo importante en el trade-off sesgo-varianza.  

c) Dejaremos fijo en esta primera ronda

En un tuneo grueso, dejaría fijos:

* `batch_size`
* `optimizer`
* `grad_clip_norm`
* `epochs`
* `patience`
* `bidirectional`

Esto reduce el espacio de búsqueda y hace más interpretable el resultado. El libro insiste en que el proceso de diseño y tuning debe ser iterativo y productivo, no una búsqueda infinita de combinaciones.

d) Valores sugeridos para la primera ronda

```python
gru_params_stage1 = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],
    "learning_rate": [1e-3, 3e-4, 1e-4],
    "weight_decay": [0.0, 1e-5, 1e-4],
}
```

e) Configuración fija sugerida para esta etapa

```python
gru_fixed_stage1 = {
    "optimizer": "adamw",
    "batch_size": 2048,
    "grad_clip_norm": 1.0,
    "epochs": 50,
    "patience": 7,
    "bidirectional": False,
    "class_weight_mode": "balanced",
}
```

f) Todavía no tuneamos:

* `batch_size`
* `optimizer`
* `bidirectional`

Porque esos hiperparámetros sí importan, pero en una primera ronda suelen aportar menos información que los cinco anteriores, y complican mucho el análisis.

g) Recomendación práctica

Para el tuneo grueso, el orden sería:

1. `hidden_size`
2. `dropout`
3. `learning_rate`
4. `weight_decay`
5. `num_layers`

Ese orden tiene sentido porque primero calibramos capacidad, luego regularización, luego dinámica de optimización.

h) Conclusión

Para una primera etapa de descarte, tunearemos primero:

* `hidden_size`
* `num_layers`
* `dropout`
* `learning_rate`
* `weight_decay`

Y dejamos el resto fijo.
Es la forma más eficiente de hacer una búsqueda inicial seria, controlada y alineada con el enfoque del libro sobre generalización, control del overfitting y ajuste progresivo del modelo.  

El siguiente paso natural es definir el **espacio coarse definitivo de GRU** ya listo para correr sobre `L = [30, 60, 180]` y `target = [t2_dir_thr_90, t2_dir_thr_120]`.


# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-14 01:24:41,544 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-14 01:25:02,866 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

# Tamaños de ventana
WINDOW_SIZES = [30, 60, 90, 120, 180]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-14 01:25:03,347 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-14 01:25:03,348 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-14 01:25:03,349 | INFO | Configuración de experimento cargada
2026-04-14 01:25:03,350 | INFO | Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
2026-04-14 01:25:03,351 | INFO | Window sizes: [30, 60, 90, 120, 180]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:5]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[60]["t2_dir_thr_90"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-14 01:25:03,363 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-14 01:25:03,869 | INFO | Windows OK      : 30
2026-04-14 01:25:03,870 | INFO | Windows missing : 0
2026-04-14 01:25:03,871 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-14 01:25:03,871 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_dir_thr_90'
        - 't2_dir_thr_120'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }


### **4.4. Creación de bundles T2**

In [8]:
# --------------------------------------------------
# Crea bundles T2 para un window_size dado
# --------------------------------------------------
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundle_t2_90  : dict
    bundle_t2_120 : dict
    """

    if len(targets) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 targets T2. Recibido: {targets}"
        )

    bundle_t2_90 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[0],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    bundle_t2_120 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[1],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    # --------------------------------------------------
    # Verificación rápida
    # --------------------------------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    print(f"\nTARGET: {targets[0]}")
    print("Train :", bundle_t2_90["train"]["X"].shape, bundle_t2_90["train"]["y"].shape)
    print("Valid :", bundle_t2_90["valid"]["X"].shape, bundle_t2_90["valid"]["y"].shape)
    print("Test  :", bundle_t2_90["test"]["X"].shape,  bundle_t2_90["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_90["scaler"]).__name__)

    print(f"\nTARGET: {targets[1]}")
    print("Train :", bundle_t2_120["train"]["X"].shape, bundle_t2_120["train"]["y"].shape)
    print("Valid :", bundle_t2_120["valid"]["X"].shape, bundle_t2_120["valid"]["y"].shape)
    print("Test  :", bundle_t2_120["test"]["X"].shape,  bundle_t2_120["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_120["scaler"]).__name__)

    return bundle_t2_90, bundle_t2_120

In [9]:
#bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

Como acceder a las ventanas X e y:

```python
X_train_90 = bundle_t2_90["train"]["X"]
y_train_90 = bundle_t2_90["train"]["y"]

X_valid_120 = bundle_t2_120["valid"]["X"]
y_valid_120 = bundle_t2_120["valid"]["y"]

scaler = bundle_t2_90["scaler"]
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [10]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

In [11]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)

    Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1). Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )

In [12]:
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [13]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_dir_thr_90",
      "horizon": 90,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # --------------------------------------------------
    # Setear esperados desde TRAIN si no se dieron
    # --------------------------------------------------
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    # --------------------------------------------------
    # Ejecutar checks
    # --------------------------------------------------
    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_targets_seq2one(
    bundle_t2_90: Dict[str, Any],
    bundle_t2_120: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para ambos targets T2.
    """
    out_t2_90 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_90,
        tag="t2_90",
        verbose=verbose,
    )

    out_t2_120 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_120,
        tag="t2_120",
        verbose=verbose,
    )

    return {
        "t2_dir_thr_90": out_t2_90,
        "t2_dir_thr_120": out_t2_120,
    }

In [14]:
#sanity_outputs = run_sanity_checks_all_targets_seq2one(
#    bundle_t2_90,
#    bundle_t2_120,
#    verbose=True,
#)

In [15]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [16]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-14 01:25:06,272 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [17]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/tuning_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [18]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/tuning_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [19]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [20]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-14 01:25:11,556 | INFO | Seeds fijadas en 42


# **10. Tuning grueso**

## **10.1. Función unitaria por bundle**

In [36]:
import copy
import numpy as np
import torch
import torch.nn as nn


class GRUClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,
        num_classes: int = 3,
    ):
        super().__init__()

        gru_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)          # (batch, seq_len, hidden_size)
        last_out = out[:, -1, :]      # many-to-one
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)    # (batch, num_classes)
        return logits


def run_gru_for_bundle_seq2one(
    bundle,
    *,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 64,
    eval_batch_size: int = 64,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    class_weight="balanced",
    num_workers: int = 0,
    verbose: bool = False,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
):
    """
    Ejecuta GRU para un bundle seq2one orientado a tuning.

    - Usa TRAIN para fit
    - Usa VALID para early stopping
    - Predice SOLO en VALID
    - Devuelve predicciones y metadatos
    - Soporta labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna
    - Permite elegir optimizer y clipping de gradiente


    Luego, Ejecuta GRU para un bundle seq2one orientado a tuning fino.

    - Usa TRAIN para entrenamiento
    - Usa VALID para early stopping
    - Predice SOLO en VALID
    - No evalúa TEST (correcto para tuning)


    """

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    use_pin_memory = device == "cuda"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    # =========================
    # 3. VALIDAR SHAPES
    # =========================
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError(
            "GRU requiere tensores 3D: (n_samples, seq_len, n_features). "
            f"Recibido train={X_train.shape}, valid={X_valid.shape}"
        )

    seq_len_train, n_features_train = X_train.shape[1], X_train.shape[2]
    seq_len_valid, n_features_valid = X_valid.shape[1], X_valid.shape[2]

    if not (
        seq_len_train == seq_len_valid
        and n_features_train == n_features_valid
    ):
        raise ValueError(
            "Inconsistencia entre shapes de train/valid. "
            f"train={X_train.shape}, valid={X_valid.shape}"
        )

    n_features = n_features_train

    # =========================
    # 4. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    unknown_valid = set(np.unique(y_valid)) - set(classes_)
    if unknown_valid:
        raise ValueError(
            f"VALID contiene clases no vistas en TRAIN: {sorted(unknown_valid)}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # =========================
    # 5. CLASS WEIGHTS
    # =========================
    criterion_weight = None
    weights_by_idx = None

    if class_weight is None:
        criterion_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_classes)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_classes * count)
            for idx, count in enumerate(counts)
        }

        criterion_weight = torch.tensor(
            [weights_by_idx[idx] for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        criterion_weight = torch.tensor(
            [weights_by_idx.get(idx, 1.0) for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    else:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 6. TENSORES EN CPU
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long)

    # =========================
    # 7. DATALOADERS
    # =========================
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    valid_ds = torch.utils.data.TensorDataset(X_valid_t, y_valid_t)

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    valid_loader = torch.utils.data.DataLoader(
        valid_ds,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # =========================
    # 8. MODELO
    # =========================
    model = GRUClassifier(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)

    optimizer_name = optimizer_name.lower()

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError("optimizer_name debe ser 'adam' o 'adamw'")

    # =========================
    # 9. HELPERS
    # =========================
    def _move_batch(x):
        if device == "cuda":
            return x.to(device, non_blocking=True)
        return x.to(device)

    def compute_valid_loss():
        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                logits = model(xb)
                loss = criterion(logits, yb)

                batch_n = xb.size(0)
                valid_loss_sum += loss.item() * batch_n
                valid_count += batch_n

                del xb, yb, logits, loss
                if device == "cuda":
                    torch.cuda.empty_cache()

        return valid_loss_sum / max(valid_count, 1)

    def predict_loader(loader):
        logits_all = []

        model.eval()
        with torch.no_grad():
            for batch in loader:
                xb = batch[0]
                xb = _move_batch(xb)

                logits = model(xb)
                logits_all.append(logits.cpu())

                del xb, logits
                if device == "cuda":
                    torch.cuda.empty_cache()

        logits_all = torch.cat(logits_all, dim=0)
        return logits_all

    # =========================
    # 10. TRAIN + EARLY STOPPING
    # =========================
    best_state = None
    best_valid_loss = np.inf
    best_epoch = 0
    wait = 0
    history = []

    try:
        for epoch in range(1, epochs + 1):
            model.train()
            train_loss_sum = 0.0
            train_count = 0

            for xb, yb in train_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                optimizer.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()

                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

                optimizer.step()

                batch_n = xb.size(0)
                train_loss_sum += loss.item() * batch_n
                train_count += batch_n

                del xb, yb, logits, loss
                if device == "cuda":
                    torch.cuda.empty_cache()

            train_loss = train_loss_sum / max(train_count, 1)
            valid_loss = compute_valid_loss()

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                }
            )

            if verbose:
                print(
                    f"[Epoch {epoch:03d}] "
                    f"train_loss={train_loss:.6f} | "
                    f"valid_loss={valid_loss:.6f}"
                )

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(
                            f"[EARLY STOP] epoch={epoch} | "
                            f"best_epoch={best_epoch} | "
                            f"best_valid_loss={best_valid_loss:.6f}"
                        )
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        # =========================
        # 11. PREDICT SOLO VALID
        # =========================
        valid_logits = predict_loader(valid_loader)

        y_pred_valid_enc = valid_logits.argmax(dim=1).numpy()
        y_proba_valid = torch.softmax(valid_logits, dim=1).numpy()

        y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

        return {
            "model": model,
            "classes_": classes_,
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
            "class_weight": class_weight,
            "window_size": bundle["window_size"],
            "target": bundle["target"],
            "criterion_weight": (
                criterion_weight.detach().cpu().numpy()
                if criterion_weight is not None else None
            ),
            "weights_by_idx": weights_by_idx,
            "history": history,
            "best_valid_loss": float(best_valid_loss),
            "best_epoch": int(best_epoch),
            "y_pred_valid": y_pred_valid,
            "y_proba_valid": y_proba_valid,
            "device": device,
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "dropout": dropout,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "epochs": epochs,
            "patience": patience,
            "random_state": random_state,
            "optimizer_name": optimizer_name,
            "grad_clip_norm": grad_clip_norm,
        }

    finally:
        if device == "cuda":
            torch.cuda.empty_cache()

## **10.2. Función de evaluación sobre uno o más bundles**

In [37]:
from typing import Any, Dict, List, Sequence, Union
import gc
import pandas as pd
import torch


def eval_gru_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "gru",
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 64,
    eval_batch_size: int = 64,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight: str = "balanced",
    num_workers: int = 0,
    verbose: bool = False,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
) -> pd.DataFrame:
    """
    Evalúa GRU para uno o varios bundles seq2one y retorna
    un DataFrame consolidado SOLO para valid.

    Ajustada para tuning:
    - entrenamiento en TRAIN
    - early stopping en VALID
    - evaluación SOLO en VALID
    - liberación explícita de memoria entre bundles
    """

    # --------------------------------------------------
    # 1) Normalizar entrada a lista
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split != "valid":
        raise ValueError("Para tuning, split debe ser únicamente 'valid'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name} | "
                f"class_weight={class_weight} | "
                f"hidden_size={hidden_size} | "
                f"num_layers={num_layers} | "
                f"dropout={dropout} | "
                f"lr={learning_rate} | "
                f"wd={weight_decay} | "
                f"opt={optimizer_name} | "
                f"clip={grad_clip_norm}"
            )

        preds = None

        try:
            # ----------------------------------------------
            # 4) Entrenar + predecir SOLO valid
            # ----------------------------------------------
            preds = run_gru_for_bundle_seq2one(
                bundle,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                learning_rate=learning_rate,
                weight_decay=weight_decay,
                batch_size=batch_size,
                eval_batch_size=eval_batch_size,
                epochs=epochs,
                patience=patience,
                random_state=random_state,
                device=device,
                class_weight=class_weight,
                num_workers=num_workers,
                verbose=False,
                optimizer_name=optimizer_name,
                grad_clip_norm=grad_clip_norm,
            )

            # ----------------------------------------------
            # 5) Seleccionar y_true / y_pred SOLO valid
            # ----------------------------------------------
            y_true = bundle["valid"]["y"]

            if "y_pred_valid" not in preds:
                raise KeyError(
                    "No existe 'y_pred_valid' en la salida de "
                    "run_gru_for_bundle_seq2one"
                )

            y_pred = preds["y_pred_valid"]

            # ----------------------------------------------
            # 6) Métricas de clasificación
            # ----------------------------------------------
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split="valid",
                target=target,
                labels=[-1, 0, 1],
            )

            # ----------------------------------------------
            # 7) A DataFrame
            # ----------------------------------------------
            df_row = metrics_to_df(
                metrics,
                model=model_name,
                split="valid",
                window_size=window_size,
                target=target,
            )

            # Metadatos base
            df_row["horizon_min"] = horizon
            df_row["class_weight_mode"] = class_weight

            # Hiperparámetros de tuning
            df_row["hidden_size"] = hidden_size
            df_row["num_layers"] = num_layers
            df_row["dropout"] = dropout
            df_row["learning_rate"] = learning_rate
            df_row["weight_decay"] = weight_decay
            df_row["batch_size"] = batch_size
            df_row["eval_batch_size"] = eval_batch_size
            df_row["epochs"] = epochs
            df_row["patience"] = patience
            df_row["random_state"] = random_state
            df_row["optimizer_name"] = optimizer_name
            df_row["grad_clip_norm"] = grad_clip_norm

            # Metadatos de entrenamiento
            df_row["best_epoch"] = preds.get("best_epoch")
            df_row["best_valid_loss"] = preds.get("best_valid_loss")
            df_row["device"] = preds.get("device")
            df_row["n_params"] = hidden_size * num_layers  # proxy simple

            rows.append(df_row)

        finally:
            # ----------------------------------------------
            # 8) Liberación explícita de memoria
            # ----------------------------------------------
            if preds is not None:
                del preds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --------------------------------------------------
    # 9) Consolidar salida
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

## **10.3. Función orquestadora por `window_size`**

In [23]:
import gc
import pandas as pd
import torch


def run_gru(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "gru",
    hidden_size: int = 32,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight: str = "balanced",
    num_workers: int = 0,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
) -> pd.DataFrame:
    """
    Ejecuta GRU para una sola window_size usando SOLO VALID.

    - Entrena en TRAIN
    - Early stopping en VALID
    - Evalúa SOLO en VALID
    - Pensado exclusivamente para tuning de hiperparámetros
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = f"{model_name}_balanced"

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"GRU | TUNING | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"hidden_size    = {hidden_size}")
            print(f"num_layers     = {num_layers}")
            print(f"dropout        = {dropout}")
            print(f"learning_rate  = {learning_rate}")
            print(f"weight_decay   = {weight_decay}")
            print(f"batch_size     = {batch_size}")
            print(f"eval_batch_size= {eval_batch_size}")
            print(f"optimizer_name = {optimizer_name}")
            print(f"grad_clip_norm = {grad_clip_norm}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size}")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | model={model_name_effective}"
            )

        df_out = eval_gru_bundles(
            bundles_t2,
            split="valid",
            model_name=model_name_effective,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            verbose=verbose,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

        # --------------------------------------------------
        # 4) Orden final
        # --------------------------------------------------
        df_out = (
            df_out
            .sort_values(["window_size", "target", "horizon_min"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "target",
                        "model",
                        "hidden_size",
                        "num_layers",
                        "dropout",
                        "learning_rate",
                        "weight_decay",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **10.4. Función incremental multi-ventana**

In [24]:
from pathlib import Path
import gc
import pandas as pd
import torch


def run_gru_incremental(
    *,
    window_sizes: list[int],
    name: str = "gru",
    verbose: bool = True,
    hidden_size: int = 32,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
) -> pd.DataFrame:
    """
    Ejecuta GRU de forma incremental para múltiples window_sizes,
    asumiendo siempre class_weight='balanced'.

    Diseñada para tuning incremental:
    - una combinación fija de HP por llamada
    - agrega resultados al mismo parquet
    - hace skip si esa combinación exacta ya fue corrida para esa L
    - usa SOLO split='valid'
    """

    class_weight = "balanced"
    name_effective = f"{name}_balanced"

    metrics_dir = DRIVE_DIR / "metrics/tuning_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir esperado por corrida
    # --------------------------------------------------
    expected_combos = {
        ("t2_dir_thr_90", "valid"),
        ("t2_dir_thr_120", "valid"),
    }
    expected_model = name_effective
    expected_class_weight = "balanced"

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)
        df_existing = None

        # ----------------------------------------------
        # 3.1) Filtrar histórico de esta combinación exacta
        # ----------------------------------------------
        if not df_hist.empty:
            mask = (
                (df_hist["window_size"] == L)
                & (df_hist["model"] == expected_model)
                & (df_hist["class_weight_mode"] == expected_class_weight)
                & (df_hist["hidden_size"] == hidden_size)
                & (df_hist["num_layers"] == num_layers)
                & (df_hist["dropout"] == dropout)
                & (df_hist["learning_rate"] == learning_rate)
                & (df_hist["weight_decay"] == weight_decay)
                & (df_hist["batch_size"] == batch_size)
                & (df_hist["eval_batch_size"] == eval_batch_size)
                & (df_hist["epochs"] == epochs)
                & (df_hist["patience"] == patience)
                & (df_hist["random_state"] == random_state)
                & (df_hist["optimizer_name"] == optimizer_name)
            )

            if "grad_clip_norm" in df_hist.columns:
                if grad_clip_norm is None:
                    mask &= df_hist["grad_clip_norm"].isna()
                else:
                    mask &= (df_hist["grad_clip_norm"] == grad_clip_norm)

            df_existing = df_hist.loc[mask].copy()

            if not df_existing.empty:
                combos_done = set(zip(df_existing["target"], df_existing["split"]))
                is_complete = expected_combos.issubset(combos_done)
            else:
                is_complete = False

            if is_complete:
                if verbose:
                    print(
                        f"[SKIP] {name_effective} | L={L} | "
                        f"HP ya existe completo"
                    )
                continue

        # ----------------------------------------------
        # 3.2) Ejecutar GRU para esta window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 100)
            print(f"[RUN] {name_effective} | L={L}")
            print(
                f"hidden_size={hidden_size} | "
                f"num_layers={num_layers} | "
                f"dropout={dropout} | "
                f"learning_rate={learning_rate} | "
                f"weight_decay={weight_decay} | "
                f"batch_size={batch_size} | "
                f"eval_batch_size={eval_batch_size} | "
                f"epochs={epochs} | "
                f"patience={patience} | "
                f"optimizer_name={optimizer_name} | "
                f"grad_clip_norm={grad_clip_norm}"
            )
            print("-" * 100)

        df_L = run_gru(
            window_size=L,
            verbose=verbose,
            model_name=name,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

        # Etiqueta de familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # 3.3) Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # 3.4) Eliminar duplicados exactos por corrida
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
            "class_weight_mode",
            "hidden_size",
            "num_layers",
            "dropout",
            "learning_rate",
            "weight_decay",
            "batch_size",
            "eval_batch_size",
            "epochs",
            "patience",
            "random_state",
            "optimizer_name",
        ]

        if "grad_clip_norm" in df_hist.columns:
            subset_cols.append("grad_clip_norm")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # 3.5) Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

        # ----------------------------------------------
        # 3.6) Liberación explícita de memoria
        # ----------------------------------------------
        del df_L, df_existing
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = [
        "window_size",
        "target",
        "split",
        "horizon_min",
        "model",
        "hidden_size",
        "num_layers",
        "dropout",
        "learning_rate",
        "weight_decay",
        "batch_size",
        "random_state",
    ]

    if "optimizer_name" in df_hist.columns:
        sort_cols.append("optimizer_name")

    if "grad_clip_norm" in df_hist.columns:
        sort_cols.append("grad_clip_norm")

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

In [ ]:
from itertools import product

# ============================================================
# 1) Batch size dinámico por window_size
# ============================================================

def get_batch_size(L: int) -> int:
    L = int(L)

    if L == 30:
        return 32768   # ← punto medio óptimo
    elif L == 60:
        return 8192
    elif L == 180:
        return 2048
    else:
        return 2048


def get_eval_batch_size(L: int) -> int:
    return get_batch_size(L)


# ============================================================
# 2) Grilla gruesa de hiperparámetros para GRU
# ============================================================

param_grid = {
    "hidden_size": [64, 128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],
    "learning_rate": [1e-3, 3e-4],
    "weight_decay": [0.0, 1e-5],
}

window_sizes = [30, 60, 180]

optimizer_name = "adamw"
grad_clip_norm = 1.0
epochs = 20
patience = 5
random_state = 42
num_workers = 0
device = None
name = "gru"

# ============================================================
# 3) Preparación combos
# ============================================================

keys, values = zip(*param_grid.items())
all_combos = list(product(*values))
total_combos = len(all_combos)

print("\n" + "=" * 100)
print(f"TOTAL COMBINACIONES A PROBAR: {total_combos}")
print("=" * 100)

# ============================================================
# 4) Tuning loop
# ============================================================

for combo_idx, combo in enumerate(all_combos, start=1):
    params = dict(zip(keys, combo))

    print("\n" + "=" * 100)
    print(f"[COMBO {combo_idx} de {total_combos}] {params}")
    print("=" * 100)

    for L in window_sizes:
        bs = get_batch_size(L)
        ebs = get_eval_batch_size(L)

        print("\n" + "-" * 100)
        print(
            f"[RUN] COMBO {combo_idx}/{total_combos} | L={L} | "
            f"hidden_size={params['hidden_size']} | "
            f"num_layers={params['num_layers']} | "
            f"dropout={params['dropout']} | "
            f"learning_rate={params['learning_rate']} | "
            f"weight_decay={params['weight_decay']} | "
            f"batch_size={bs} | "
            f"eval_batch_size={ebs}"
        )
        print("-" * 100)

        df_hist = run_gru_incremental(
            window_sizes=[L],
            name=name,
            verbose=True,
            hidden_size=params["hidden_size"],
            num_layers=params["num_layers"],
            dropout=params["dropout"],
            learning_rate=params["learning_rate"],
            weight_decay=params["weight_decay"],
            batch_size=bs,
            eval_batch_size=ebs,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

print("\n" + "=" * 100)
print("[DONE] Tuning grueso GRU finalizado")
print("=" * 100)


TOTAL COMBINACIONES A PROBAR: 32

[COMBO 1 de 32] {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 0.0}

----------------------------------------------------------------------------------------------------
[RUN] COMBO 1/32 | L=30 | hidden_size=64 | num_layers=1 | dropout=0.1 | learning_rate=0.001 | weight_decay=0.0 | batch_size=16384 | eval_batch_size=16384
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
[RUN] gru_balanced | L=30
hidden_size=64 | num_layers=1 | dropout=0.1 | learning_rate=0.001 | weight_decay=0.0 | batch_size=16384 | eval_batch_size=16384 | epochs=20 | patience=5 | optimizer_name=adamw | grad_clip_norm=1.0
----------------------------------------------------------------------------------------------------

GRU | TUNING | WINDOW_SIZE=L30
hidden_size    = 64
num_layers     =

2026-04-13 00:57:11,716 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-13 00:57:11,717 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-13 00:57:12,526 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-13 00:57:12,526 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-13 00:57:13,145 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-13 00:57:13,146 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-13 00:57:13,436 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-13 00:57:13,437 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-13 00:57:14,819 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-13 00:57:14,819 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-13 00:57:15,615 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-13 00:57:15,616 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-13 00:57:16,271 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=64 | num_layers=1 | dropout=0.1 | lr=0.001 | wd=0.0 | opt=adamw | clip=1.0


KeyboardInterrupt: 

In [25]:
from itertools import product

# ============================================================
# 1) Batch size más conservador para terminar los faltantes
# ============================================================

def get_batch_size(L: int) -> int:
    L = int(L)

    if L == 30:
        return 16384   # bajado para evitar OOM
    elif L == 60:
        return 8192
    elif L == 180:
        return 2048
    else:
        return 2048


def get_eval_batch_size(L: int) -> int:
    return get_batch_size(L)


# ============================================================
# 2) Misma grilla ya usada
# ============================================================

param_grid = {
    "hidden_size": [64, 128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],
    "learning_rate": [1e-3, 3e-4],
    "weight_decay": [0.0, 1e-5],
}

window_sizes = [30, 60, 180]

optimizer_name = "adamw"
grad_clip_norm = 1.0
epochs = 20
patience = 5
random_state = 42
num_workers = 0
device = None
name = "gru"

# ============================================================
# 3) Preparación combos
# ============================================================

keys, values = zip(*param_grid.items())
all_combos = list(product(*values))
total_combos = len(all_combos)

print("\n" + "=" * 100)
print(f"TOTAL COMBINACIONES EN LA GRILLA: {total_combos}")
print("REANUDANDO SOLO DESDE EL COMBO 25 AL 32")
print("=" * 100)

# ============================================================
# 4) Reanudar solo combos 25..32
#    enumerate empieza en 1, así que usamos slicing [24:]
# ============================================================

start_combo = 25

for combo_idx, combo in enumerate(all_combos[start_combo - 1 :], start=start_combo):
    params = dict(zip(keys, combo))

    print("\n" + "=" * 100)
    print(f"[COMBO {combo_idx} de {total_combos}] {params}")
    print("=" * 100)

    for L in window_sizes:
        bs = get_batch_size(L)
        ebs = get_eval_batch_size(L)

        print("\n" + "-" * 100)
        print(
            f"[RUN] COMBO {combo_idx}/{total_combos} | L={L} | "
            f"hidden_size={params['hidden_size']} | "
            f"num_layers={params['num_layers']} | "
            f"dropout={params['dropout']} | "
            f"learning_rate={params['learning_rate']} | "
            f"weight_decay={params['weight_decay']} | "
            f"batch_size={bs} | "
            f"eval_batch_size={ebs}"
        )
        print("-" * 100)

        df_hist = run_gru_incremental(
            window_sizes=[L],
            name=name,
            verbose=True,
            hidden_size=params["hidden_size"],
            num_layers=params["num_layers"],
            dropout=params["dropout"],
            learning_rate=params["learning_rate"],
            weight_decay=params["weight_decay"],
            batch_size=bs,
            eval_batch_size=ebs,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

print("\n" + "=" * 100)
print("[DONE] Reanudación de tuning GRU completada (combos 25 al 32)")
print("=" * 100)


TOTAL COMBINACIONES EN LA GRILLA: 32
REANUDANDO SOLO DESDE EL COMBO 25 AL 32

[COMBO 25 de 32] {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.1, 'learning_rate': 0.001, 'weight_decay': 0.0}

----------------------------------------------------------------------------------------------------
[RUN] COMBO 25/32 | L=30 | hidden_size=128 | num_layers=2 | dropout=0.1 | learning_rate=0.001 | weight_decay=0.0 | batch_size=16384 | eval_batch_size=16384
----------------------------------------------------------------------------------------------------
[SKIP] gru_balanced | L=30 | HP ya existe completo

----------------------------------------------------------------------------------------------------
[RUN] COMBO 25/32 | L=60 | hidden_size=128 | num_layers=2 | dropout=0.1 | learning_rate=0.001 | weight_decay=0.0 | batch_size=8192 | eval_batch_size=8192
----------------------------------------------------------------------------------------------------
[SKIP] gru_balanced | L=60 | HP ya exi

In [26]:
df_hist

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,...,eval_batch_size,epochs,patience,random_state,optimizer_name,grad_clip_norm,best_epoch,best_valid_loss,device,family
0,gru_balanced,valid,30,t2_dir_thr_120,93508,0.381213,0.381178,0.637054,0.687545,0.427690,...,32768,20,5,42,adamw,1.0,5,1.084676,cuda,gru_balanced
1,gru_balanced,valid,30,t2_dir_thr_120,93508,0.381213,0.381178,0.637054,0.687545,0.427690,...,32768,20,5,42,adamw,1.0,5,1.084676,cuda,gru_balanced
2,gru_balanced,valid,30,t2_dir_thr_120,93508,0.417171,0.412521,0.614167,0.602761,0.411540,...,32768,20,5,42,adamw,1.0,10,1.080145,cuda,gru_balanced
3,gru_balanced,valid,30,t2_dir_thr_120,93508,0.417171,0.412521,0.614167,0.602761,0.411540,...,32768,20,5,42,adamw,1.0,10,1.080145,cuda,gru_balanced
4,gru_balanced,valid,30,t2_dir_thr_120,93508,0.380172,0.379704,0.636385,0.687107,0.425692,...,32768,20,5,42,adamw,1.0,5,1.084554,cuda,gru_balanced
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,gru_balanced,valid,180,t2_dir_thr_90,64408,0.431485,0.426437,0.570412,0.558223,0.424900,...,2048,20,5,42,adamw,1.0,3,1.073469,cuda,gru_balanced
188,gru_balanced,valid,180,t2_dir_thr_90,64408,0.434193,0.429083,0.573484,0.563051,0.426668,...,2048,20,5,42,adamw,1.0,3,1.083339,cuda,gru_balanced
189,gru_balanced,valid,180,t2_dir_thr_90,64408,0.434193,0.429083,0.573484,0.563051,0.426668,...,2048,20,5,42,adamw,1.0,3,1.083339,cuda,gru_balanced
190,gru_balanced,valid,180,t2_dir_thr_90,64408,0.432880,0.427342,0.571504,0.559263,0.425010,...,2048,20,5,42,adamw,1.0,3,1.076542,cuda,gru_balanced


# **11. Análisis de tuning grueso**

## 11.1. Ranking global de GRU

In [27]:
df_rank = (
    df_hist
    .sort_values("bal_acc_gain_vs_naive", ascending=False)
    .reset_index(drop=True)
)

df_rank.head(20)

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,...,eval_batch_size,epochs,patience,random_state,optimizer_name,grad_clip_norm,best_epoch,best_valid_loss,device,family
0,gru_balanced,valid,30,t2_dir_thr_90,93508,0.438353,0.435367,0.635931,0.632556,0.435393,...,32768,20,5,42,adamw,1.0,8,1.058941,cuda,gru_balanced
1,gru_balanced,valid,30,t2_dir_thr_90,93508,0.438353,0.435367,0.635931,0.632556,0.435393,...,32768,20,5,42,adamw,1.0,8,1.058941,cuda,gru_balanced
2,gru_balanced,valid,180,t2_dir_thr_90,64408,0.435769,0.434951,0.587826,0.589088,0.435896,...,2048,20,5,42,adamw,1.0,3,1.080618,cuda,gru_balanced
3,gru_balanced,valid,180,t2_dir_thr_90,64408,0.435769,0.434951,0.587826,0.589088,0.435896,...,2048,20,5,42,adamw,1.0,3,1.080618,cuda,gru_balanced
4,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435677,0.432194,0.632738,0.627251,0.431013,...,16384,20,5,42,adamw,1.0,6,1.065155,cuda,gru_balanced
5,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435677,0.432194,0.632738,0.627251,0.431013,...,16384,20,5,42,adamw,1.0,6,1.065155,cuda,gru_balanced
6,gru_balanced,valid,180,t2_dir_thr_90,64408,0.435532,0.435006,0.587061,0.586775,0.434899,...,2048,20,5,42,adamw,1.0,3,1.080833,cuda,gru_balanced
7,gru_balanced,valid,180,t2_dir_thr_90,64408,0.435532,0.435006,0.587061,0.586775,0.434899,...,2048,20,5,42,adamw,1.0,3,1.080833,cuda,gru_balanced
8,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435216,0.427636,0.632813,0.633807,0.435459,...,16384,20,5,42,adamw,1.0,14,1.071887,cuda,gru_balanced
9,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435216,0.427636,0.632813,0.633807,0.435459,...,16384,20,5,42,adamw,1.0,14,1.071887,cuda,gru_balanced


In [28]:
df_rank.columns

Index(['model', 'split', 'window_size', 'target', 'n_samples',
       'balanced_accuracy', 'f1_macro', 'f1_weighted', 'accuracy',
       'precision_macro', 'recall_macro', 'balanced_accuracy_naive',
       'bal_acc_gain_vs_naive', 'horizon_min', 'class_weight_mode',
       'hidden_size', 'num_layers', 'dropout', 'learning_rate', 'weight_decay',
       'batch_size', 'eval_batch_size', 'epochs', 'patience', 'random_state',
       'optimizer_name', 'grad_clip_norm', 'best_epoch', 'best_valid_loss',
       'device', 'family'],
      dtype='object')

a) Dominancia clara de configuración

* Las mejores configuraciones se concentran en:

  * `window_size = 30`
  * `target = t2_dir_thr_90`

Conclusión:
La señal predictiva está dominada por patrones de corto plazo y horizontes más inmediatos.

---

b) Ventanas cortas superan consistentemente a largas

* `L=30` aparece sistemáticamente en el top
* `L=60` queda en zona intermedia
* `L=180` rinde peor

Conclusión:
El modelo GRU pierde capacidad predictiva al aumentar la longitud de la secuencia, probablemente por ruido acumulado.

---

c) Sensibilidad a la complejidad del modelo

* Configuraciones con:

  * `hidden_size = 128`
  * `num_layers = 2`

no muestran mejoras claras sobre modelos más simples

Conclusión:
Aumentar la capacidad del modelo no aporta ganancia significativa. El problema no requiere alta complejidad temporal.

---

d) Estabilidad del rendimiento

* `balanced_accuracy ≈ 0.433 – 0.438`
* Variación muy baja entre configuraciones top

Conclusión:
El modelo es estable frente a cambios de hiperparámetros. Existe una meseta de rendimiento.

---

e) Impacto limitado de `dropout`

* Aparecen tanto:

  * `dropout = 0.1`
  * `dropout = 0.3`

en configuraciones competitivas

Conclusión:
La regularización por dropout no es un factor determinante en este problema.

---

f) Impacto moderado de `learning_rate`

* Las mejores configuraciones incluyen:

  * `learning_rate = 1e-3`

Conclusión:
Un learning rate relativamente alto funciona bien. No se observa necesidad de ajustes más finos en esta etapa.

---

g) Early stopping temprano

* `best_epoch` típicamente bajo:

  * valores entre 3 y 14

Conclusión:
El modelo converge rápidamente. Entrenamientos largos no aportan valor adicional.

---

h) Duplicación de resultados

* Se observan filas duplicadas con mismas métricas

Conclusión:
Esto se debe al cambio de `batch_size` durante la ejecución. No afecta el análisis, pero deberá limpiarse en la siguiente etapa.

---

i) Patrón ganador

Configuración representativa:

* `window_size = 30`
* `target = t2_dir_thr_90`
* `hidden_size = 64–128`
* `num_layers = 1`
* `learning_rate = 1e-3`

---

j) Interpretación del modelo

El GRU está capturando:

* dependencias temporales cortas
* relaciones no lineales simples
* patrones intradía inmediatos

No está aprovechando memoria larga, lo cual es consistente con el dominio.

---

k) Conclusión general

* Existe una región óptima clara en el espacio de hiperparámetros
* El modelo no requiere alta complejidad
* La señal es estable pero con menor ganancia que modelos tipo boosting

El problema queda listo para:

* reducción del espacio de búsqueda
* tuning fino en una región más acotada
* comparación final contra otros modelos seleccionados



## 11.2. Análisis por hiperparámetro


Objetivo: identificar qué hiperparámetros están aportando realmente a la mejora del modelo y qué valores muestran mejor desempeño promedio durante el tuning grueso.


In [ ]:
# ============================================================
# 11.2. Análisis por hiperparámetro - GRU balanced
# Objetivo:
# identificar qué hiperparámetros están aportando realmente
# a la mejora del modelo y qué valores muestran mejor
# desempeño promedio durante el tuning grueso.
# ============================================================

metric = "balanced_accuracy"

print("\n" + "=" * 80)
print("ANÁLISIS POR HIPERPARÁMETRO | GRU_BALANCED")
print("=" * 80)

# ------------------------------------------------------------
# 0) Filtro base
#    Solo valid + gru_balanced
# ------------------------------------------------------------
df_gru = df_hist.copy()

df_gru = df_gru[
    (df_gru["model"] == "gru_balanced") &
    (df_gru["split"] == "valid")
].copy()

print(f"\nFilas analizadas: {len(df_gru)}")

# ------------------------------------------------------------
# 1) hidden_size
# ------------------------------------------------------------
print("\n[1] hidden_size")
res_hidden_size = (
    df_gru
    .groupby("hidden_size")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_hidden_size)

# ------------------------------------------------------------
# 2) num_layers
# ------------------------------------------------------------
print("\n[2] num_layers")
res_num_layers = (
    df_gru
    .groupby("num_layers")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_num_layers)

# ------------------------------------------------------------
# 3) dropout
# ------------------------------------------------------------
print("\n[3] dropout")
res_dropout = (
    df_gru
    .groupby("dropout")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_dropout)

# ------------------------------------------------------------
# 4) learning_rate
# ------------------------------------------------------------
print("\n[4] learning_rate")
res_learning_rate = (
    df_gru
    .groupby("learning_rate")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_learning_rate)

# ------------------------------------------------------------
# 5) weight_decay
# ------------------------------------------------------------
print("\n[5] weight_decay")
res_weight_decay = (
    df_gru
    .groupby("weight_decay")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_weight_decay)

# ------------------------------------------------------------
# 6) batch_size
#    opcional: útil para ver si afectó el tuning
# ------------------------------------------------------------
print("\n[6] batch_size")
res_batch_size = (
    df_gru
    .groupby("batch_size")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_batch_size)

# ------------------------------------------------------------
# 7) best_epoch
#    útil para analizar convergencia
# ------------------------------------------------------------
print("\n[7] best_epoch")
res_best_epoch = (
    df_gru
    .groupby("best_epoch")[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
)
display(res_best_epoch)


ANÁLISIS POR HIPERPARÁMETRO | GRU_BALANCED

Filas analizadas: 192

[1] hidden_size


,mean,median,max,count
hidden_size,,,,
128,0.420879,0.426732,0.438353,96
64,0.413747,0.419041,0.434687,96



[2] num_layers


,mean,median,max,count
num_layers,,,,
2,0.419163,0.424269,0.435677,96
1,0.415463,0.422800,0.438353,96



[3] dropout


...

,mean,median,max,count
weight_decay,,,,
0.00000,0.417313,0.423899,0.438353,96
0.00001,0.417313,0.423899,0.438353,96



[6] batch_size


,mean,median,max,count
batch_size,,,,
2048,0.429618,0.430770,0.435769,64
16384,0.422196,0.418126,0.435677,16
32768,0.414514,0.421081,0.438353,48
8192,0.405885,0.406390,0.434453,64



[7] best_epoch


,mean,median,max,count
best_epoch,,,,
14,0.435216,0.435216,0.435216,2
16,0.433053,0.433053,0.433053,2
3,0.429694,0.431853,0.435769,24
7,0.428636,0.428636,0.428636,2
12,0.428099,0.432763,0.435118,6
8,0.427035,0.421891,0.438353,6
4,0.425284,0.428206,0.430213,16
18,0.423708,0.423708,0.423708,2
10,0.422450,0.417171,0.433907,6


a) Ventaja clara de `hidden_size = 128`

* Se observa:

  * `128 → mean = 0.4209`
  * `64 → mean = 0.4137`

Conclusión:
Un mayor tamaño oculto mejora consistentemente el desempeño. El modelo se beneficia de mayor capacidad representacional.

---

b) Leve ventaja de mayor profundidad (`num_layers = 2`)

* Resultados:

  * `2 capas → mean = 0.4192`
  * `1 capa → mean = 0.4155`

Conclusión:
Agregar una segunda capa aporta mejora, pero moderada. No es un factor dominante, pero sí consistente.

---

c) Bajo impacto de `dropout`

* Resultados muy cercanos:

  * `0.1 → mean = 0.4179`
  * `0.3 → mean = 0.4168`

Conclusión:
La regularización mediante dropout no tiene un impacto relevante en este problema.

---

d) Fuerte impacto de `learning_rate`

* Diferencia clara:

  * `0.001 → mean = 0.4224`
  * `0.0003 → mean = 0.4122`

Conclusión:
Un learning rate más alto funciona mejor. El modelo necesita actualizaciones más agresivas para capturar la señal.

---

e) `weight_decay` no aporta valor

* Resultados idénticos:

  * `0.0` y `1e-5` → mismo desempeño

Conclusión:
La regularización L2 no tiene impacto en este contexto.

---

f) Efecto del `batch_size`

* Mejores resultados:

  * `2048 → mean = 0.4296` (mejor)
  * `16384 → 0.4222`
  * `32768 → 0.4145`
  * `8192 → 0.4059`

Conclusión:
Batch más pequeños generalizan mejor. Batch grandes degradan el desempeño.

---

g) Convergencia temprana (`best_epoch`)

* Mejores resultados concentrados en:

  * epochs bajos (2–6 principalmente)
  * algunos picos en epochs más altos (14–16), pero con muy pocos casos

Conclusión:
El modelo converge rápidamente. Entrenamientos largos no aportan mejora consistente.

---

h) Patrón dominante

Configuración favorecida:

* `hidden_size = 128`
* `num_layers = 2`
* `learning_rate = 0.001`
* `dropout = 0.1` (ligera preferencia)
* `batch_size` pequeño (idealmente cercano a 2048)

---

i) Interpretación del modelo

El GRU:

* requiere capacidad moderada (hidden_size alto)
* se beneficia de cierta profundidad
* pero no necesita regularización fuerte
* ni entrenamiento prolongado

---

j) Conclusión general

* Los hiperparámetros más relevantes son:

  * `hidden_size`
  * `learning_rate`
  * `batch_size`

* Los menos relevantes:

  * `dropout`
  * `weight_decay`

* El modelo muestra:

  * buena estabilidad
  * rápida convergencia
  * sensibilidad a capacidad y dinámica de entrenamiento

Esto deja definido un espacio claro para el tuning fino.


## 11.3. Análisis por ventana

Objetivo: validar si la ventana corta (L=30) sigue siendo la mejor escala temporal tras el tuning grueso.

In [29]:
df_hist.groupby("window_size")["bal_acc_gain_vs_naive"] \
      .agg(["mean", "median", "max", "count"]) \
      .sort_values("mean", ascending=False)

,mean,median,max,count
window_size,,,,
180,0.096285,0.097437,0.102436,64
30,0.083101,0.086944,0.105019,64
60,0.072552,0.073057,0.101120,64


a) La ventana L=180 presenta el mayor rendimiento promedio (mean ≈ 0.096), lo que indica que, en términos agregados, captura mejor la señal que las demás escalas temporales.

b) Sin embargo, L=30 alcanza el mayor valor máximo (max ≈ 0.105), lo que evidencia que las mejores configuraciones individuales siguen ocurriendo en ventanas cortas.

c) La mediana también favorece a L=180, lo que sugiere mayor estabilidad y consistencia en comparación con L=30, cuya dispersión es más alta.

d) L=60 es claramente inferior en todas las métricas (mean, median, max), por lo que puede descartarse como ventana relevante.

e) Conclusión:
- L=180 → más robusta y consistente
- L=30 → mayor potencial pero más variable

f) Implicación práctica:
No hay un único ganador absoluto:
- si se prioriza robustez → L=180
- si se prioriza performance máxima → L=30

## 11.4. Análisis por target

Objetivo: comparar el desempeño entre los horizontes de predicción (`t2_dir_thr_90` vs `t2_dir_thr_120`) para identificar cuál presenta mayor separabilidad.

In [30]:
df_hist.groupby("target")["bal_acc_gain_vs_naive"] \
      .agg(["mean", "median", "max", "count"]) \
      .sort_values("mean", ascending=False)


,mean,median,max,count
target,,,,
t2_dir_thr_90,0.092355,0.096470,0.105019,96
t2_dir_thr_120,0.075603,0.081072,0.098443,96


a) El target t2_dir_thr_90 presenta un rendimiento superior en todas las métricas:
- mayor mean (≈ 0.092 vs 0.075)
- mayor mediana (≈ 0.096 vs 0.081)
- mayor máximo (≈ 0.105 vs 0.098)

b) La diferencia en mean (~0.017) es significativa, lo que indica que el horizonte de 90 minutos es sistemáticamente más predecible.

c) La mediana también es claramente mayor, lo que confirma que no es un efecto de outliers sino una mejora consistente.

d) El target t2_dir_thr_120 muestra menor separabilidad, lo que sugiere mayor ruido o menor señal en horizontes más largos.

e) Conclusión:
- t2_dir_thr_90 es el target dominante en términos de señal y estabilidad.
- t2_dir_thr_120 puede descartarse como candidato principal.

f) Implicación práctica:
El tuning fino debe centrarse en:
- target: t2_dir_thr_90

## 11.5. Análisis conjunto (modelo + L + target + hiperparámetros)


Objetivo: identificar configuraciones realmente robustas, es decir, combinaciones específicas de hiperparámetros que consistentemente producen alto rendimiento.


In [32]:
# ============================================================
# 11.5. Análisis conjunto (GRU)
# Objetivo:
# identificar configuraciones robustas (modelo + L + target + HP)
# ============================================================

metric = "balanced_accuracy"

print("\n" + "=" * 80)
print("ANÁLISIS CONJUNTO | GRU_BALANCED")
print("=" * 80)

# ------------------------------------------------------------
# 0) Filtro base
# ------------------------------------------------------------
df_gru = df_hist.copy()

df_gru = df_gru[
    (df_gru["model"] == "gru_balanced") &
    (df_gru["split"] == "valid")
].copy()

print(f"\nFilas analizadas: {len(df_gru)}")

# ------------------------------------------------------------
# 1) Columnas relevantes (configuración completa)
# ------------------------------------------------------------
cols = [
    "window_size",
    "target",
    "hidden_size",
    "num_layers",
    "dropout",
    "learning_rate",
    "weight_decay",
]

# ------------------------------------------------------------
# 2) Ranking por configuración
# ------------------------------------------------------------
df_top = (
    df_gru
    .groupby(cols)[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

# ------------------------------------------------------------
# 3) Top configuraciones
# ------------------------------------------------------------
display(df_top.head(15))


ANÁLISIS CONJUNTO | GRU_BALANCED

Filas analizadas: 192


,window_size,target,hidden_size,num_layers,dropout,learning_rate,weight_decay,mean,median,max,count
0,30,t2_dir_thr_90,128,1,0.1,0.0010,0.00001,0.438353,0.438353,0.438353,1
1,30,t2_dir_thr_90,128,1,0.1,0.0010,0.00000,0.438353,0.438353,0.438353,1
2,180,t2_dir_thr_90,128,1,0.3,0.0010,0.00000,0.435769,0.435769,0.435769,1
3,180,t2_dir_thr_90,128,1,0.3,0.0010,0.00001,0.435769,0.435769,0.435769,1
4,30,t2_dir_thr_90,128,2,0.1,0.0010,0.00000,0.435677,0.435677,0.435677,1
5,30,t2_dir_thr_90,128,2,0.1,0.0010,0.00001,0.435677,0.435677,0.435677,1
6,180,t2_dir_thr_90,128,1,0.1,0.0010,0.00001,0.435532,0.435532,0.435532,1
7,180,t2_dir_thr_90,128,1,0.1,0.0010,0.00000,0.435532,0.435532,0.435532,1
8,30,t2_dir_thr_90,128,2,0.1,0.0003,0.00000,0.435216,0.435216,0.435216,1
9,30,t2_dir_thr_90,128,2,0.1,0.0003,0.00001,0.435216,0.435216,0.435216,1


In [34]:
cols_reduced = [
    "window_size",
    "target",
    "hidden_size",
    "num_layers",
    "learning_rate",
]

df_top_reduced = (
    df_gru
    .groupby(cols_reduced)[metric]
    .agg(["mean", "median", "max", "count"])
    .sort_values("mean", ascending=False)
    .reset_index()
)
print('Top_reduced:')
display(df_top_reduced.head(15))

Top_reduced:


,window_size,target,hidden_size,num_layers,learning_rate,mean,median,max,count
0,180,t2_dir_thr_90,128,1,0.0010,0.435650,0.435650,0.435769,4
1,30,t2_dir_thr_90,128,2,0.0010,0.434651,0.434651,0.435677,4
2,180,t2_dir_thr_90,64,2,0.0010,0.434143,0.434143,0.434687,4
3,30,t2_dir_thr_90,128,1,0.0003,0.433940,0.433940,0.435118,4
4,180,t2_dir_thr_90,64,2,0.0003,0.433926,0.433926,0.433967,4
5,180,t2_dir_thr_90,64,1,0.0010,0.433363,0.433363,0.433369,4
6,180,t2_dir_thr_90,128,2,0.0003,0.433206,0.433206,0.434193,4
7,180,t2_dir_thr_90,128,2,0.0010,0.432182,0.432182,0.432880,4
8,30,t2_dir_thr_90,128,1,0.0010,0.432125,0.432125,0.438353,4
9,180,t2_dir_thr_120,128,1,0.0010,0.431528,0.431528,0.431776,4


a) Dominancia clara de `target = t2_dir_thr_90`

* En ambos rankings (completo y reducido), casi todas las configuraciones top corresponden a:

  * `t2_dir_thr_90`

Conclusión:
Se confirma que el horizonte de 90 minutos es el más predecible y consistente.

---

b) Fuerte presencia de `hidden_size = 128`

* En el top:

  * mayoría de configuraciones usan `128`
  * `64` aparece pero con menor frecuencia

Conclusión:
La capacidad del modelo es un factor clave. El modelo necesita mayor dimensionalidad interna.

---

c) Preferencia por `learning_rate = 0.001`

* En el ranking reducido:

  * `0.001` domina las primeras posiciones
  * `0.0003` aparece, pero generalmente por debajo

Conclusión:
Un aprendizaje más agresivo mejora el rendimiento en GRU.

---

d) Ventanas cortas siguen siendo relevantes, pero no exclusivas

* Aparece:

  * `L=30` en varias configuraciones top
  * pero también `L=180` en el primer lugar

Conclusión:
A diferencia de XGB, GRU logra capturar señal también en ventanas largas, aunque sin ventaja clara.

---

e) Bajo impacto de `dropout` y `weight_decay`

* En el ranking completo:

  * mismas métricas con `weight_decay = 0` y `1e-5`
  * `dropout` alterna entre 0.1 y 0.3 sin patrón claro

Conclusión:
Estos hiperparámetros no son determinantes.

---

f) Profundidad del modelo no es decisiva

* `num_layers = 1` y `2` aparecen en el top

Conclusión:
No hay una ventaja clara por mayor profundidad.

---

g) Configuración óptima representativa

* `window_size = 30`
* `target = t2_dir_thr_90`
* `hidden_size = 128`
* `num_layers = 1–2`
* `learning_rate = 0.001`

---

h) Diferencias de rendimiento pequeñas

* Las mejores configuraciones están en un rango estrecho:

  * ~0.432 – 0.438

Conclusión:
El modelo presenta una meseta de rendimiento. No hay un “salto” claro entre configuraciones.

---

i) Conclusión general

* Se confirma una región óptima clara

* Los hiperparámetros más influyentes son:

  * `hidden_size`
  * `learning_rate`

* El resto tiene impacto limitado

Esto deja definido un espacio acotado y bien estructurado para el tuning fino.


## 11.6. Decisiones a tomar a partir del tuning grueso


Objetivo: consolidar una región óptima para GRU, reducir el espacio de búsqueda y definir un tuning fino eficiente.

---

1. Fijar región óptima

A partir del análisis del tuning grueso:

* `window_size = 30` (principal), con soporte secundario en `60` y `180`

* `target = t2_dir_thr_90`

* `hidden_size = 128`

* `num_layers = 1–2` (sin diferencia clara)

* `learning_rate = 0.001`

* `dropout = 0.1–0.3` (indiferente)

* `weight_decay = 0.0` (sin impacto relevante)

Conclusión:

* El modelo requiere **capacidad moderada (hidden_size alto)**
* La señal sigue siendo de corto plazo
* No hay beneficio claro en aumentar profundidad ni regularización

Esta región define la base del siguiente paso.

---

2. Reducir espacio de búsqueda

Se pasa de un grid amplio (~32 combinaciones) a un espacio más enfocado:

Ejemplo de reducción:

* `hidden_size`: [128]
* `num_layers`: [1, 2]
* `learning_rate`: [0.001]
* `dropout`: [0.1, 0.2] (opcional)
* `weight_decay`: [0.0]

Eliminando:

* `hidden_size = 64`
* `learning_rate = 0.0003`
* combinaciones redundantes de regularización

Conclusión:

Se concentra el tuning en la zona donde el modelo ya mostró mejor desempeño.

---

3. Definir tuning fino

En GRU, el tuning fino no se centra tanto en arquitectura, sino en dinámica de entrenamiento:

Hiperparámetros a ajustar:

* `hidden_size` (refinamiento)

  * [128, 192, 256]
  * explorar si mayor capacidad aporta mejora marginal

* `learning_rate` (ajuste fino)

  * [0.001, 0.0007, 0.0005]
  * balance entre convergencia y estabilidad

* `batch_size`

  * clave para generalización
  * explorar valores más bajos (ej: 2048–8192)

* `dropout`

  * ajuste fino en [0.1, 0.2, 0.3]

* `grad_clip_norm`

  * estabilidad del entrenamiento
  * valores típicos: [0.5, 1.0, 2.0]

Opcional:

* scheduler de learning rate (si se implementa)

Conclusión:

El tuning fino en GRU apunta a:

* mejorar estabilidad del entrenamiento
* optimizar convergencia
* ajustar capacidad del modelo

No a cambios estructurales grandes.

---

4. Conclusión general

* Existe una región óptima clara y consistente
* El modelo requiere capacidad moderada pero no complejidad excesiva
* La señal es estable, pero con menor ganancia que modelos tipo boosting

El siguiente paso es:

* refinar hiperparámetros de entrenamiento
* reducir variabilidad
* mejorar consistencia del modelo

Esto permite un tuning fino:

* más eficiente
* más controlado
* enfocado en mejoras reales y no exploración amplia


## 11.7. Resultado esperado del análisis


A partir de los resultados obtenidos, la región óptima real para GRU en este problema es:

Mejor región:

* `window_size`: 30 (principal), con desempeño competitivo en 60 y 180

* `target`: t2_dir_thr_90

* `hidden_size`: 128

* `num_layers`: 1–2 (sin diferencia crítica)

* `learning_rate`: 0.001

* `dropout`: 0.1–0.3 (indiferente)

* `weight_decay`: 0.0 (sin impacto)

* `batch_size`: bajo–medio (≈ 2048–8192 óptimo para generalización)

Observación:

A diferencia de configuraciones genéricas de deep learning, este problema no requiere:

* redes profundas (muchas capas)
* regularización fuerte
* entrenamiento prolongado

La mejor performance se logra con:

* modelos de capacidad moderada
* entrenamiento relativamente agresivo (learning rate alto)
* ventanas cortas
* batches moderados (no extremos)



## 11.8. Conclusión


Este análisis permite responder con claridad:

qué hiperparámetros importan:

* `hidden_size`
* `learning_rate`
* `batch_size` (impacto en generalización)

qué hiperparámetros tienen bajo impacto:

* `dropout`
* `weight_decay`
* `num_layers` (en este rango)

qué valores funcionan:

* capacidad media (`hidden_size = 128`)
* learning rate relativamente alto (`0.001`)
* ventana corta
* target de corto horizonte (`t2_dir_thr_90`)

qué configuraciones son robustas:

* aquellas con métricas estables (mean ≈ median)
* consistentes frente a cambios de regularización
* con convergencia temprana (early stopping bajo)

Conclusión final del proceso:

* La región óptima está claramente definida
* El problema presenta una señal intradía de corto plazo
* El GRU captura dependencias temporales simples, no de largo alcance
* El modelo no requiere alta complejidad para capturar la señal

Este punto deja el pipeline listo para:

tuning fino, enfocado en:

* ajuste de capacidad (hidden_size)
* estabilidad del entrenamiento
* optimización de batch_size y learning_rate

como última etapa antes de la comparación final entre modelos.



```python
param_grid_fino = {
    # arquitectura (ya casi fija)
    "hidden_size": [128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],

    # dinámica de entrenamiento
    "learning_rate": [1e-3],
    "batch_size": [2048, 4096, 8192],
    "grad_clip_norm": [0.5, 1.0, 2.0],

    # descartado
    "weight_decay": [0.0],
}

window_sizes = [30]
targets = ["t2_dir_thr_90"]
```



# **12. Tuning Fino**

## **12.1. `run_gru_tuning_fino`**


In [41]:
import gc
import pandas as pd
import torch


def run_gru_tuning_fino(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "gru",
    hidden_size: int = 32,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight: str = "balanced",
    num_workers: int = 0,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
) -> pd.DataFrame:
    """
    Ejecuta GRU para tuning fino (solo target óptimo).

    - window_size fijo por ejecución
    - target: t2_dir_thr_90
    - entrenamiento en TRAIN
    - early stopping en VALID
    - evaluación SOLO en VALID
    """

    size = int(window_size)

    bundle_t2_90 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = f"{model_name}_balanced"

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"GRU | TUNING FINO | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"hidden_size    = {hidden_size}")
            print(f"num_layers     = {num_layers}")
            print(f"dropout        = {dropout}")
            print(f"learning_rate  = {learning_rate}")
            print(f"weight_decay   = {weight_decay}")
            print(f"batch_size     = {batch_size}")
            print(f"eval_batch_size= {eval_batch_size}")
            print(f"optimizer_name = {optimizer_name}")
            print(f"grad_clip_norm = {grad_clip_norm}")

        # --------------------------------------------------
        # 2) Construcción de bundles (SOLO t2_dir_thr_90)
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | target=t2_dir_thr_90")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # SOLO usar el target 90
        bundles_t2 = [bundle_t2_90]

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | model={model_name_effective}"
            )

        df_out = eval_gru_bundles(
            bundles_t2,
            split="valid",
            model_name=model_name_effective,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            verbose=verbose,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

        # --------------------------------------------------
        # 4) Orden final
        # --------------------------------------------------
        df_out = (
            df_out
            .sort_values(["window_size", "target", "horizon_min"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "target",
                        "model",
                        "hidden_size",
                        "num_layers",
                        "dropout",
                        "learning_rate",
                        "weight_decay",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **12.2. `run_gru_incremental_fine`**

In [42]:
from pathlib import Path
import gc
import pandas as pd
import torch


def run_gru_incremental_fine(
    *,
    window_sizes: list[int],
    name: str = "gru",
    verbose: bool = True,
    hidden_size: int = 32,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
    optimizer_name: str = "adamw",
    grad_clip_norm: float | None = 1.0,
) -> pd.DataFrame:

    class_weight = "balanced"
    name_effective = f"{name}_balanced"

    # ============================================================
    # 📁 NUEVA RUTA PARA TUNING FINO
    # ============================================================
    metrics_dir = DRIVE_DIR / "metrics/tuning_metrics/fine_tuning"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = metrics_dir / f"classification_{name_effective}_fine.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definición de esperado (solo 90)
    # --------------------------------------------------
    expected_combos = {
        ("t2_dir_thr_90", "valid"),
    }

    # --------------------------------------------------
    # 3) Loop por ventanas
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        df_existing = None

        if not df_hist.empty:
            mask = (
                (df_hist["window_size"] == L)
                & (df_hist["model"] == name_effective)
                & (df_hist["class_weight_mode"] == class_weight)
                & (df_hist["hidden_size"] == hidden_size)
                & (df_hist["num_layers"] == num_layers)
                & (df_hist["dropout"] == dropout)
                & (df_hist["learning_rate"] == learning_rate)
                & (df_hist["weight_decay"] == weight_decay)
                & (df_hist["batch_size"] == batch_size)
                & (df_hist["eval_batch_size"] == eval_batch_size)
                & (df_hist["epochs"] == epochs)
                & (df_hist["patience"] == patience)
                & (df_hist["random_state"] == random_state)
                & (df_hist["optimizer_name"] == optimizer_name)
            )

            if "grad_clip_norm" in df_hist.columns:
                if grad_clip_norm is None:
                    mask &= df_hist["grad_clip_norm"].isna()
                else:
                    mask &= (df_hist["grad_clip_norm"] == grad_clip_norm)

            df_existing = df_hist.loc[mask].copy()

            if not df_existing.empty:
                combos_done = set(zip(df_existing["target"], df_existing["split"]))
                is_complete = expected_combos.issubset(combos_done)
            else:
                is_complete = False

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name_effective} | L={L} | HP ya existe")
                continue

        # --------------------------------------------------
        # 4) Ejecutar modelo (SOLO target 90)
        # --------------------------------------------------
        if verbose:
            print("\n" + "-" * 100)
            print(f"[RUN] {name_effective} | L={L}")
            print(
                f"hidden_size={hidden_size} | "
                f"num_layers={num_layers} | "
                f"dropout={dropout} | "
                f"learning_rate={learning_rate} | "
                f"batch_size={batch_size} | "
                f"grad_clip_norm={grad_clip_norm}"
            )
            print("-" * 100)

        df_L = run_gru_tuning_fino(
            window_size=L,
            verbose=verbose,
            model_name=name,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
        )

        df_L["family"] = name_effective

        # --------------------------------------------------
        # 5) Append histórico
        # --------------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # --------------------------------------------------
        # 6) Eliminar duplicados
        # --------------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "hidden_size",
            "num_layers",
            "dropout",
            "learning_rate",
            "weight_decay",
            "batch_size",
            "eval_batch_size",
            "epochs",
            "patience",
            "random_state",
            "optimizer_name",
        ]

        if "grad_clip_norm" in df_hist.columns:
            subset_cols.append("grad_clip_norm")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 7) Guardar
        # --------------------------------------------------
        df_hist.to_parquet(metrics_path, index=False)

        # --------------------------------------------------
        # 8) Liberar memoria
        # --------------------------------------------------
        del df_L, df_existing
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------
    # 9) Return final
    # --------------------------------------------------
    return df_hist.sort_values(
        ["window_size", "target", "model", "hidden_size"]
    ).reset_index(drop=True)

## **12.3. Ejecución de tuning fino**

In [43]:
from itertools import product

# ============================================================
# 1) Batch size FIJO (ya se tunea como hiperparámetro)
# ============================================================

def get_eval_batch_size(bs: int) -> int:
    return bs


# ============================================================
# 2) Grilla de tuning fino
# ============================================================

param_grid_fino = {
    "hidden_size": [128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],

    "learning_rate": [1e-3],
    "batch_size": [2048, 4096, 8192],
    "grad_clip_norm": [0.5, 1.0, 2.0],

    "weight_decay": [0.0],
}

window_sizes = [30]

optimizer_name = "adamw"
epochs = 20
patience = 5
random_state = 42
num_workers = 0
device = None
name = "gru"


# ============================================================
# 3) Preparación combos
# ============================================================

keys, values = zip(*param_grid_fino.items())
all_combos = list(product(*values))
total_combos = len(all_combos)

print("\n" + "=" * 100)
print(f"TOTAL COMBINACIONES TUNING FINO: {total_combos}")
print("=" * 100)


# ============================================================
# 4) Loop principal
# ============================================================

for combo_idx, combo in enumerate(all_combos, start=1):
    params = dict(zip(keys, combo))

    print("\n" + "=" * 100)
    print(f"[COMBO {combo_idx} de {total_combos}] {params}")
    print("=" * 100)

    for L in window_sizes:
        bs = params["batch_size"]
        ebs = get_eval_batch_size(bs)

        print("\n" + "-" * 100)
        print(
            f"[RUN] COMBO {combo_idx}/{total_combos} | L={L} | "
            f"hidden_size={params['hidden_size']} | "
            f"num_layers={params['num_layers']} | "
            f"dropout={params['dropout']} | "
            f"learning_rate={params['learning_rate']} | "
            f"batch_size={bs} | "
            f"grad_clip_norm={params['grad_clip_norm']}"
        )
        print("-" * 100)

        df_hist = run_gru_incremental_fine(
            window_sizes=[L],
            name=name,
            verbose=True,
            hidden_size=params["hidden_size"],
            num_layers=params["num_layers"],
            dropout=params["dropout"],
            learning_rate=params["learning_rate"],
            weight_decay=params["weight_decay"],
            batch_size=bs,
            eval_batch_size=ebs,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=params["grad_clip_norm"],
        )


print("\n" + "=" * 100)
print("[DONE] Tuning fino GRU finalizado")
print("=" * 100)


TOTAL COMBINACIONES TUNING FINO: 36

[COMBO 1 de 36] {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 2048, 'grad_clip_norm': 0.5, 'weight_decay': 0.0}

----------------------------------------------------------------------------------------------------
[RUN] COMBO 1/36 | L=30 | hidden_size=128 | num_layers=1 | dropout=0.1 | learning_rate=0.001 | batch_size=2048 | grad_clip_norm=0.5
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
[RUN] gru_balanced | L=30
hidden_size=128 | num_layers=1 | dropout=0.1 | learning_rate=0.001 | batch_size=2048 | grad_clip_norm=0.5
----------------------------------------------------------------------------------------------------

GRU | TUNING FINO | WINDOW_SIZE=L30
hidden_size    = 128
num_layers     = 1
dropout        = 0.1
learning_rate  = 0.001
weight_decay 

2026-04-14 01:54:08,474 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 01:54:08,475 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 01:54:09,154 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 01:54:09,155 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 01:54:09,660 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 01:54:09,660 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 01:54:10,020 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 01:54:10,021 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 01:54:11,402 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 01:54:11,403 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 01:54:11,909 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 01:54:11,910 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 01:54:12,348 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=128 | num_layers=1 | dropout=0.1 | lr=0.001 | wd=0.0 | opt=adamw | clip=0.5

[DONE] L30 | rows=1
 window_size        target        model  hidden_size  num_layers  dropout  learning_rate  weight_decay  balanced_accuracy  f1_macro
          30 t2_dir_thr_90 gru_balanced          128           1      0.1          0.001           0.0           0.436502  0.437034

[COMBO 2 de 36] {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 2048, 'grad_clip_norm': 1.0, 'weight_decay': 0.0}

---------------------

2026-04-14 01:56:13,598 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 01:56:13,598 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 01:56:13,743 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 01:56:13,744 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 01:56:13,889 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 01:56:13,890 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 01:56:13,893 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 01:56:13,894 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 01:56:14,542 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 01:56:14,543 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 01:56:14,700 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 01:56:14,701 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 01:56:14,855 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=128 | num_layers=1 | dropout=0.1 | lr=0.001 | wd=0.0 | opt=adamw | clip=1.0

[DONE] L30 | rows=1
 window_size        target        model  hidden_size  num_layers  dropout  learning_rate  weight_decay  balanced_accuracy  f1_macro
          30 t2_dir_thr_90 gru_balanced          128           1      0.1          0.001           0.0           0.431711  0.428348

[COMBO 3 de 36] {'hidden_size': 128, 'num_layers': 1, 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 2048, 'grad_clip_norm': 2.0, 'weight_decay': 0.0}

---------------------

...


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=128 | num_layers=2 | dropout=0.3 | lr=0.001 | wd=0.0 | opt=adamw | clip=0.5

[DONE] L30 | rows=1
 window_size        target        model  hidden_size  num_layers  dropout  learning_rate  weight_decay  balanced_accuracy  f1_macro
          30 t2_dir_thr_90 gru_balanced          128           2      0.3          0.001           0.0           0.437189  0.429424

[COMBO 35 de 36] {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 8192, 'grad_clip_norm': 1.0, 'weight_decay': 0.0}

--------------------

2026-04-14 02:45:37,599 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 02:45:37,600 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 02:45:37,738 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 02:45:37,738 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 02:45:37,893 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 02:45:37,894 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 02:45:37,898 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 02:45:37,898 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 02:45:38,554 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 02:45:38,554 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 02:45:38,699 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 02:45:38,700 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 02:45:38,852 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=128 | num_layers=2 | dropout=0.3 | lr=0.001 | wd=0.0 | opt=adamw | clip=1.0

[DONE] L30 | rows=1
 window_size        target        model  hidden_size  num_layers  dropout  learning_rate  weight_decay  balanced_accuracy  f1_macro
          30 t2_dir_thr_90 gru_balanced          128           2      0.3          0.001           0.0           0.437189  0.429424

[COMBO 36 de 36] {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 8192, 'grad_clip_norm': 2.0, 'weight_decay': 0.0}

--------------------

2026-04-14 02:47:35,859 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-14 02:47:35,860 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 02:47:36,014 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-14 02:47:36,015 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 02:47:36,171 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-14 02:47:36,172 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-14 02:47:36,177 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-14 02:47:36,177 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-14 02:47:36,891 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-14 02:47:36,891 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-14 02:47:37,045 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-14 02:47:37,046 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-14 02:47:37,186 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=gru_balanced
  -> L30 | target=t2_dir_thr_90 | split=valid | model=gru_balanced | class_weight=balanced | hidden_size=128 | num_layers=2 | dropout=0.3 | lr=0.001 | wd=0.0 | opt=adamw | clip=2.0

[DONE] L30 | rows=1
 window_size        target        model  hidden_size  num_layers  dropout  learning_rate  weight_decay  balanced_accuracy  f1_macro
          30 t2_dir_thr_90 gru_balanced          128           2      0.3          0.001           0.0           0.437189  0.429424

[DONE] Tuning fino GRU finalizado


In [44]:
df_hist

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,...,epochs,patience,random_state,optimizer_name,grad_clip_norm,best_epoch,best_valid_loss,device,n_params,family
0,gru_balanced,valid,30,t2_dir_thr_90,93508,0.436502,0.437034,0.642407,0.646832,0.440150,...,20,5,42,adamw,0.5,7,1.069775,cuda,128,gru_balanced
1,gru_balanced,valid,30,t2_dir_thr_90,93508,0.431711,0.428348,0.633987,0.634352,0.430275,...,20,5,42,adamw,1.0,4,1.069876,cuda,128,gru_balanced
2,gru_balanced,valid,30,t2_dir_thr_90,93508,0.431711,0.428348,0.633987,0.634352,0.430275,...,20,5,42,adamw,2.0,4,1.069876,cuda,128,gru_balanced
3,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435144,0.432861,0.632887,0.627551,0.431177,...,20,5,42,adamw,0.5,2,1.059019,cuda,128,gru_balanced
4,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435144,0.432861,0.632887,0.627551,0.431177,...,20,5,42,adamw,1.0,2,1.059019,cuda,128,gru_balanced
5,gru_balanced,valid,30,t2_dir_thr_90,93508,0.435144,0.432861,0.632887,0.627551,0.431177,...,20,5,42,adamw,2.0,2,1.059019,cuda,128,gru_balanced
6,gru_balanced,valid,30,t2_dir_thr_90,93508,0.438450,0.436505,0.636929,0.633529,0.435283,...,20,5,42,adamw,0.5,4,1.063286,cuda,128,gru_balanced
7,gru_balanced,valid,30,t2_dir_thr_90,93508,0.438450,0.436505,0.636929,0.633529,0.435283,...,20,5,42,adamw,1.0,4,1.063286,cuda,128,gru_balanced
8,gru_balanced,valid,30,t2_dir_thr_90,93508,0.438450,0.436505,0.636929,0.633529,0.435283,...,20,5,42,adamw,2.0,4,1.063286,cuda,128,gru_balanced
9,gru_balanced,valid,30,t2_dir_thr_90,93508,0.431687,0.428807,0.633266,0.631390,0.429100,...,20,5,42,adamw,0.5,4,1.066672,cuda,128,gru_balanced


**1. Mejora marginal pero consistente**

* El mejor valor alcanza:

  * `balanced_accuracy ≈ 0.4404`

Comparado con el tuning grueso (~0.438):

👉 mejora real, pero pequeña (~+0.002)

Conclusión:
El modelo ya estaba cerca del óptimo → el tuning fino solo ajusta.

---

**2. Dominancia de `hidden_size = 128` vs 256**

* `hidden_size = 128` → ~0.435–0.439
* `hidden_size = 256` → resultados mixtos, pero:

  * algunos muy buenos (~0.4404)
  * otros muy malos (~0.421)

Conclusión:

* 256 tiene **mayor varianza**
* 128 es **más estable**

👉 trade-off clásico: capacidad vs estabilidad

---

**3. `grad_clip_norm` NO es determinante**

Se repite patrón:

* 0.5, 1.0 y 2.0 → mismos resultados en varios casos

Ejemplo claro:

```text
0.440452 (0.5, 1.0, 2.0)
```

Conclusión:

* el modelo ya está en régimen estable
* clipping no está limitando

👉 pierde importancia en esta región

---

**4. Importancia de `batch_size`**

Los mejores resultados aparecen en configuraciones con:

* batch relativamente pequeño (según tu grid)

Conclusión:

✔ se confirma:
batch chico → mejor generalización

---

**5. `num_layers` no aporta mejora clara**

* No hay dominio fuerte de 2 capas
* 1 capa compite perfectamente

Conclusión:

👉 mayor profundidad **no agrega señal**

---

**6. `dropout` bajo sigue siendo competitivo**

* No hay evidencia fuerte de que 0.3 mejore
* 0.1 aparece en múltiples configuraciones top

Conclusión:

👉 regularización ligera es suficiente

---

**7. Patrón ganador**

Configuraciones top (~0.440):

* `window_size = 30`
* `target = t2_dir_thr_90`
* `hidden_size = 128 o 256`
* `num_layers = 1`
* `dropout = 0.1`
* `learning_rate = 1e-3`
* `batch_size = 2048–4096`
* `grad_clip_norm = irrelevante`

---

**8. Insight clave (muy importante)**

El modelo muestra:

* sensibilidad a capacidad (`hidden_size`)
* poca sensibilidad a:

  * clipping
  * profundidad
  * regularización

👉 esto indica:

✔ la señal ya es “fácil” de aprender
✔ el modelo no necesita mucha sofisticación adicional

---

**9. Conclusión general**

* La región óptima está completamente definida
* El tuning fino confirma lo encontrado en el grueso
* Las mejoras son marginales pero reales
* El modelo ya está cerca de su techo de performance

---

**10. Decisión práctica**

Puedes fijar:

```python
hidden_size = 128
num_layers = 1
dropout = 0.1
learning_rate = 1e-3
batch_size = 2048 o 4096
grad_clip_norm = 1.0 (por estabilidad estándar)
```


# **13. Modelo GRU — Configuración final óptima**



Modelo GRU — Configuración final óptima

Objetivo: Clasificación T2 (dirección con umbral)

---

Configuración óptima



```python
window_size = 30  
target = t2_dir_thr_90  

hidden_size = 128  
num_layers = 1  
dropout = 0.1  

learning_rate = 1e-3  
batch_size = 2048  
grad_clip_norm = 1.0  

optimizer = AdamW  
epochs = 20  
patience = 5  
class_weight = balanced  

```
---

Desempeño esperado

- balanced_accuracy ≈ 0.440  
- Mejora consistente sobre baseline (≈ +0.10 vs naive)  
- Buen equilibrio entre precisión y recall (f1_macro estable)  

---

Características del modelo

- Arquitectura simple y eficiente  
- Baja profundidad (1 capa)  
- Capacidad moderada (hidden_size = 128)  
- Regularización ligera (dropout = 0.1)  
- Entrenamiento estable (grad clipping + early stopping)  

---

Interpretación

El modelo captura:

- señales intradía de corto plazo  
- patrones temporales simples  
- relaciones no lineales moderadas  

No requiere:

- alta profundidad  
- gran capacidad  
- regularización compleja  

---

Robustez

- Resultados consistentes en múltiples combinaciones  
- Baja sensibilidad a hiperparámetros secundarios  
- Estabilidad entre entrenamiento y validación  

---

Conclusión final

- La región óptima está claramente definida  
- El modelo es simple, estable y suficiente para el problema  
- El rendimiento alcanzado está cercano al techo para esta arquitectura  

---

Estado del modelo

- Listo para comparación final contra otros modelos  
- No requiere más tuning estructural  
- Solo ajustes marginales posibles  